In [1]:
import sys
sys.path.append('..')
from osp import *
import html

In [40]:
df_feats = STASH_SLICE_FEATS.df
df_feats

,pos_DT,pos_VBZ,pos_RB,pos_JJ,pos_IN,pos_NN,pos_PRP$,pos_NNS,pos_TO,pos_VB,...,pos_LS,pos_NFP,deprel_vocative,pos_SYM,deprel_orphan,pos_AFX,pos_ADD,deprel_goeswith,pos_GW,deprel_obl:tmod
_key,,,,,,,,,,,,,,,,,,,,,
phil/10.2307/2380200__04,84.104289,46.257359,81.581161,116.904962,111.017662,102.607233,15.979815,45.416316,22.708158,58.031960,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/2514892__02,87.415222,20.346647,16.578749,97.211756,143.180106,113.790505,5.275057,66.314996,5.275057,11.303693,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/14710__03,102.811245,11.244980,46.586345,100.401606,130.120482,167.068273,12.048193,52.208835,8.032129,17.670683,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/20111773__02,98.193244,56.559309,56.559309,98.193244,135.113904,109.190888,6.284368,35.349568,9.426551,24.351925,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
phil/10.2307/2379466__05,91.644205,32.345013,65.588500,79.065588,139.263252,141.958670,11.680144,53.908356,12.578616,40.431267,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
lit/468421__06,116.161616,40.404040,44.612795,75.757576,136.363636,162.457912,11.784512,54.713805,14.309764,37.878788,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
other/10.2307/29782027__24,80.839895,18.372703,29.921260,51.968504,70.341207,87.664042,0.524934,25.721785,7.349081,25.721785,...,NaN,5.249344,NaN,0.524934,NaN,NaN,NaN,NaN,NaN,NaN
lit/459531__04,92.471358,40.098200,36.824877,80.196399,135.842881,134.206219,18.003273,49.099836,21.276596,32.733224,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [71]:
df_feats_z = pd.DataFrame(df_feats.values, index=df_feats.index, columns=df_feats.columns)
for c in df_feats_z.columns:
    df_feats_z[c] = pd.to_numeric(df_feats_z[c], errors='coerce').fillna(0)
    df_feats_z[c] = (df_feats_z[c] - df_feats_z[c].mean()) / df_feats_z[c].std()
# df_feats_z

In [72]:
feats = ['deprel_cop', 'pos_MD', 'deprel_aux', 'deprel_expl']
top_slices = df_feats_z[feats].mean(axis=1).sort_values(ascending=False)
top_slices.index[0]

'phil/10.2307/4544720__03'

In [73]:
slice_id = 'phil/10.2307/4544720__03'

In [74]:
df_meta = get_corpus_metadata()


In [ ]:

def get_slice_html_simple(slice_id, **kwargs):
    text_id = slice_id.split('__')[0]
    text_row = df_meta.loc[text_id]
    feat_row = STASH_SLICE_FEATS.get(slice_id)
    out = [f'<h3><a href="https://{text_row.url}">{text_row.author}, “{text_row.title},” <i>{text_row.journal}</i> ({text_row.year})</a> [{slice_id}]</h3>']
    out.append('<ul>')

    feats = []
    for k in kwargs:
        if k.startswith('feat_'):
            v = kwargs[k]
            if isinstance(v, str):
                feats.append(v)
            elif isinstance(v, dict):
                for k2 in v:
                    for vv in v[k2]:
                        feats.append(f'{k2}_{vv}')
    
    for f in feats:
        print(df_feats_z.loc)
        fz = df_feats_z.loc[slice_id, f]
        out.append(f'<li><b>{f}</b>: {feat_row[f]:.1f} / 1000 words = {"+" if fz > 0 else ""}{fz:.1f}z</li>')
    out.append('</ul>')

    doc = stanza.Document.from_serialized(STASH_SLICES_NLP[slice_id])
    sents = doc.sentences
    htmlx=get_sents_html_simple(sents, **kwargs)
    htmlx1 = htmlx.replace('<ol>','').replace('</ol>','').replace('<li>','').replace('</li>','')
    # out.append(htmlx1)
    out.append(htmlx)
    out.append(f'<pre>{html_to_latex(htmlx)}</pre>')
    return '\n'.join(out)

def get_slices_html_simple(slice_ids, **kwargs):
    out = []
    for slice_id in slice_ids:
        out.append(get_slice_html_simple(slice_id, **kwargs))
        out.append('<hr>')
    return '\n'.join(out)



In [76]:
htmlx = get_slice_html_simple(slice_id, feat_bold='deprel_cop', feat_italic='pos_MD', feat_underline={'deprel':['expl','aux']})
HTML(htmlx)